<a href="https://colab.research.google.com/github/heberdavi/mba-engsoft-tcc/blob/main/notebooks/01-processamento_pln.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

📑 Guia de Execução Estratégica
⚠️ IMPORTANTE: Sempre que o Runtime (Ambiente de Execução) for reiniciado, as Células 1 e 2 devem ser executadas obrigatoriamente para restabelecer os caminhos do Drive e reinstalar as bibliotecas.

🔄 Fluxo de Dependências:
Sessão Recém-Iniciada: Executar Célula 1 ➔ Célula 2.

Primeira vez no projeto: Executar Célula 1 ➔ Célula 2 ➔ Célula 3 (Carga).

Retomando Processamento: Se o banco já existe no Drive, pule a Célula 3 e vá direto para a Célula 4 e/ou 5 e/ou 6.

In [ ]:
# Célula 1: Montagem do Google Drive e Configuração de Caminhos
from google.colab import drive
import os

# 1. Montagem Segura: Só executa se ainda não estiver montado
if not os.path.exists('/content/drive/MyDrive'):
    print("📂 Montando Google Drive...")
    drive.mount('/content/drive')
else:
    print("✅ Google Drive já está montado e acessível.")

# 2. Configuração Estrita de Caminhos
DRIVE_DIR = '/content/drive/MyDrive/mba-engsof-tcc/versao_pos_entrega'
DB_FILE_NAME = 'data/base-dados.db'
DB_PATH = os.path.join(DRIVE_DIR, DB_FILE_NAME)

# Artefatos SQL
SCHEMA_SQL = os.path.join(DRIVE_DIR, 'sql/01-schema.sql')
SEED_SQL = os.path.join(DRIVE_DIR, 'sql/02-seed_data.sql')

# Pasta de Saída (Outputs)
EXPORT_PATH = os.path.join(DRIVE_DIR, 'outputs')
if not os.path.exists(EXPORT_PATH):
    os.makedirs(EXPORT_PATH)
    print(f"📁 Pasta de exportação criada em: {EXPORT_PATH}")

print(f"📍 Banco de Dados: {DB_PATH}")

In [ ]:
# Célula 2: Instalação das bibliotecas e inicialização da estrutura (Schema)

# 1. Instalação Silenciosa
!pip install -q transformers torch pandas bertopic pysentimiento spacy
!python -m spacy download pt_core_news_lg -q

import sqlite3
import torch

# 2. Hardware Check para BERTimbau/BERTopic
device = 0 if torch.cuda.is_available() else -1

def inicializar_estrutura_db(db_path, schema_path):
    """Garante que a estrutura de tabelas esteja presente."""
    print(f"🛠️ Verificando integridade das tabelas...")

    # Se o arquivo de banco não existir, o SQLite o criará automaticamente
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    try:
        with open(schema_path, 'r', encoding='utf-8') as f:
            cursor.executescript(f.read())
        conn.commit()
        print("✅ Estrutura (Schema) validada com sucesso!")
    except Exception as e:
        print(f"❌ Erro ao processar Schema: {e}")
    finally:
        conn.close()

# 3. Execução
inicializar_estrutura_db(DB_PATH, SCHEMA_SQL)

print(f"\n🚀 Ambiente pronto (GPU: {'Ativa' if device == 0 else 'Inativa'}).")

In [ ]:
# Célula 3: Carga Inicial de Dados (Seed SQL)
def executar_carga_dados(db_path, seed_path):
    """Popula o banco apenas se a tabela 'verso' estiver vazia."""
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    try:
        # Verifica se já existem dados para evitar duplicidade no Drive
        cursor.execute("SELECT count(*) FROM verso")
        total_existente = cursor.fetchone()[0]

        if total_existente > 0:
            print(f"ℹ️ O banco já contém {total_existente} versos. Carga inicial ignorada.")
            return

        print("🌱 Semeando dados iniciais (02-seed_data.sql)... Isso pode levar alguns minutos.")
        with open(seed_path, 'r', encoding='utf-8') as f:
            cursor.executescript(f.read())

        conn.commit()
        print(f"✅ Carga de {seed_path} concluída com sucesso!")

    except sqlite3.OperationalError as e:
        print(f"⚠️ Erro operacional: {e}. Verifique se a Célula 2 foi executada.")
    except Exception as e:
        print(f"❌ Erro crítico na carga: {e}")
    finally:
        conn.close()

# Executa a carga (Somente se necessário)
executar_carga_dados(DB_PATH, SEED_SQL)

In [ ]:
# Célula 4: Indexação Hierárquica e Sensores XAI (Versão Otimizada com Cache)
import spacy
import sqlite3
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm

# 1. Preparação do Modelo NLP
try:
    nlp = spacy.load("pt_core_news_lg")
except:
    import os
    os.system("python -m spacy download pt_core_news_lg")
    nlp = spacy.load("pt_core_news_lg")

def executar_indexacao_completa(db_path):
    conn = None
    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()

        # --- PREPARAÇÃO DO BANCO ---
        print("🧹 Limpando índices anteriores...")
        cursor.execute("DELETE FROM verso_palavra")
        cursor.execute("DELETE FROM palavra")
        cursor.execute("DELETE FROM verso_limpo")
        conn.commit()

        # Carrega palavras existentes para o cache em memória (evita query por token)
        cursor.execute("SELECT lemma, id FROM palavra")
        dicionario_palavras = dict(cursor.fetchall())

        # 2. Carga dos Versos ativos para processamento
        query = """
            SELECT v.id, v.texto
            FROM verso v
            WHERE v.processar = 'S'
        """
        df_versos = pd.read_sql_query(query, conn)
        print(f"🧠 Analisando {len(df_versos)} versos...")

        for _, row in tqdm(df_versos.iterrows(), total=len(df_versos), desc="Gramática e Sensores"):
            verso_id = row['id']
            texto = row['texto']

            if not texto or len(texto.strip()) < 3:
                continue

            doc = nlp(texto)
            total_tokens = len(doc)
            if total_tokens == 0:
                continue

            # Acumuladores de Metadados
            counts = {'ADJ': 0, 'ADV': 0, 'PROPN': 0, 'VERB': 0, 'NUM': 0, 'NOUN': 0}
            n_primeira_pessoa = 0
            palavras_sig_len = []
            is_identidade = 0
            tem_numeral = 0

            # --- PROCESSAMENTO POR TOKEN ---
            for t in doc:
                lemma_lower = t.lemma_.lower()
                pos_atual = t.pos_
                is_stop_int = 1 if t.is_stop else 0

                # Gestão de Cache da Tabela 'palavra'
                if lemma_lower not in dicionario_palavras:
                    cursor.execute("""
                        INSERT OR IGNORE INTO palavra (lemma, pos_tag, is_stop)
                        VALUES (?, ?, ?)
                    """, (lemma_lower, pos_atual, is_stop_int))

                    cursor.execute("SELECT id FROM palavra WHERE lemma = ?", (lemma_lower,))
                    resultado_id = cursor.fetchone()
                    if resultado_id:
                        dicionario_palavras[lemma_lower] = resultado_id[0]

                palavra_id = dicionario_palavras.get(lemma_lower)

                # Sensores Morfossilógicos
                if pos_atual in counts:
                    counts[pos_atual] += 1
                if t.morph.get("Person") == ["1"]:
                    n_primeira_pessoa += 1
                if pos_atual == 'NUM':
                    tem_numeral = 1

                # Sensor de Identidade (Ontologia: 'ser'/'estar' como raiz ou cópula)
                if lemma_lower in ['ser', 'estar'] and (t.dep_ in ['ROOT', 'cop']):
                    is_identidade = 1

                if not t.is_stop and not t.is_punct:
                    palavras_sig_len.append(len(t.text))

                # Índice Invertido com Hierarquia
                if palavra_id:
                    cursor.execute("""
                        INSERT INTO verso_palavra (
                            verso_id, palavra_id, posicao, head_pos, dep_relation, morph
                        ) VALUES (?, ?, ?, ?, ?, ?)
                    """, (verso_id, palavra_id, t.i, t.head.i, t.dep_, str(t.morph)))

            # --- MÉTRICAS DE NÍVEL MACRO ---
            score_emocional = (counts['ADJ'] + counts['ADV']) / total_tokens
            score_informativo = (counts['PROPN'] + counts['NUM']) / total_tokens
            score_acao = counts['VERB'] / total_tokens

            # Linha corrigida com a checagem limpa da lista
            avg_word_len = np.mean(palavras_sig_len) if palavras_sig_len else 0.0


            # Entropia Gramatical Estável
            present_tags = [v for v in counts.values() if v > 0]
            total_tags_contadas = sum(present_tags)
            if total_tags_contadas > 0:
                entropia = -sum([(v/total_tags_contadas) * np.log(v/total_tags_contadas + 1e-9) for v in present_tags])
            else:
                entropia = 0.0

            # Texto Lematizado para o Modelo Zero-Shot
            texto_limpo = " ".join([t.lemma_.lower() for t in doc if not t.is_stop and not t.is_punct])

            # Persistência dos Sensores XAI
            cursor.execute("""
                INSERT INTO verso_limpo (
                    verso_id, texto_limpo, score_emocional, score_informativo,
                    score_acao, entropia_gramatical, n_primeira_pessoa,
                    avg_word_len, is_identidade, tem_numeral
                ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """, (verso_id, texto_limpo, score_emocional, score_informativo,
                  score_acao, entropia, n_primeira_pessoa, avg_word_len,
                  is_identidade, tem_numeral))

        conn.commit()
        print("✅ Sucesso: Banco de dados atualizado e persistido.")

    except sqlite3.OperationalError as e:
        print(f"⚠️ Erro de Bloqueio (Lock): {e}")
        if conn: conn.rollback()
    except Exception as e:
        print(f"❌ Erro inesperado: {e}")
        if conn: conn.rollback()
    finally:
        if conn:
            conn.close()
            print("🔒 Conexão fechada com segurança.")

# Execução automática direcionada à sua constante global
executar_indexacao_completa(DB_PATH)

In [ ]:
# Célula 5: Classificação por Eixos Existenciais (IA + DNA Sintático + Salvaguarda de Transição)
import sqlite3
import pandas as pd
import numpy as np
from transformers import pipeline
from tqdm.notebook import tqdm

# 1. Configuração e Carga de Dados conforme o Modelo de Dados
conn = sqlite3.connect(DB_PATH)

# Recupera Eixos e as Sentenças de Descrição Qualificadas
query_eixos = """
    SELECT e.id, e.nome, GROUP_CONCAT(ed.sentenca, ' ') as descricao_completa
    FROM eixo e
    JOIN eixo_descricao ed ON e.id = ed.eixo_id
    GROUP BY e.id ORDER BY e.id
"""
df_eixos = pd.read_sql_query(query_eixos, conn)

# Mapeamentos estruturados para isolar o risco de desordem no SQL
label_para_id = dict(zip(df_eixos['descricao_completa'], df_eixos['id']))
labels_ia = df_eixos['descricao_completa'].tolist()

# Query integrada: Versos + Metadados XAI + Gênero Literário
query_input = """
    SELECT
        vl.*,
        gl.nome as genero_nome
    FROM verso_limpo vl
    JOIN verso v ON v.id = vl.verso_id
    JOIN livro l ON l.id = v.livro_id
    JOIN genero_literario gl ON l.genero_id = gl.id
    WHERE v.processar = 'S'
"""
df_input = pd.read_sql_query(query_input, conn)

# 2. Inicialização do Modelo BART (Zero-Shot Classification)
classifier = pipeline("zero-shot-classification",
                      model="facebook/bart-large-mnli",
                      device=0)

def classificar_hibrido(row, labels_ia, df_eixos, label_para_id):
    # --- 1. IDENTIFICAÇÃO DINÂMICA DO EIXO NARRATIVO ---
    row_narrativo = df_eixos[df_eixos['nome'].str.contains('Narrativo|Histórico', case=False, na=False)]
    id_narrativo = row_narrativo['id'].iloc[0] if not row_narrativo.empty else None

    # --- 2. SALVAGUARDA PARA VERSOS CURTOS (Bypass da IA) ---
    palavras = str(row['texto_limpo']).split()
    verbos_elocucao = ['dizer', 'responder', 'continuar', 'falar', 'clamar', 'acrescentar', 'perguntar', 'replicar']

    # Heurística: Se o texto limpo tem até 2 palavras e contém verbos de elocução, força Eixo Narrativo
    if len(palavras) <= 2 and any(v in row['texto_limpo'] for v in verbos_elocucao):
        ids_ordenados = sorted(label_para_id.values())
        # Atribui probabilidade 1.0 ao eixo narrativo e 0.0 aos demais
        probs_finais = [1.0 if eixo_id == id_narrativo else 0.0 for eixo_id in ids_ordenados]
        return probs_finais, ids_ordenados, "Bypass: Transição Dialética"

    # --- 3. FLUXO NORMAL: ENGENHARIA DE PROMPT SINTÁTICA (XAI) ---
    dna = []
    if row['n_primeira_pessoa'] > 0: dna.append("Relato Pessoal/Subjetivo")
    if row['is_identidade'] == 1: dna.append("Definição de Identidade/Estado")
    if row['tem_numeral'] == 1: dna.append("Dados Quantitativos/Inventário")

    prefixo = f"[Contexto: {', '.join(dna)}] " if dna else ""
    texto_para_ia = prefixo + row['texto_limpo']

    # Inferência profunda da IA
    res = classifier(texto_para_ia, labels_ia, multi_label=False)

    # Monta mapa temporário { ID_DO_EIXO: SCORE_IA }
    scores_por_id = {}
    for label, score in zip(res['labels'], res['scores']):
        eixo_id = label_para_id[label]
        scores_por_id[eixo_id] = score

    # --- 4. REGRAS DE SOBERANIA LITERÁRIA ---
    genero = row['genero_nome']

    if id_narrativo in scores_por_id:
        if genero == 'Poético/Sapiencial':
            # Se houver alta densidade de dados na moldura, bonifica o Narrativo
            if row['score_informativo'] > 0.12 or row['tem_numeral'] == 1:
                scores_por_id[id_narrativo] += 0.45
            else:
                # No corpo poético corrido, penaliza o narrativo para priorizar reflexões existenciais
                scores_por_id[id_narrativo] -= 0.15
        elif genero == 'Epístola':
            scores_por_id[id_narrativo] -= 0.25

    # Re-normalização estrita baseada na ordem sequencial dos IDs
    ids_ordenados = sorted(scores_por_id.keys())
    probs_vetor = [scores_por_id[eixo_id] for eixo_id in ids_ordenados]

    probs_clipped = np.clip(probs_vetor, 0.001, 1.0)
    soma_completude = sum(probs_clipped)
    probs_finais = [p / soma_completude for p in probs_clipped]

    return probs_finais, ids_ordenados, "IA + DNA Sintático"

# 3. Processamento em Lote com Monitorização de Métricas de Governança
results = []
print(f"🤖 Classificando {len(df_input)} versos...")

for _, row in tqdm(df_input.iterrows(), total=len(df_input), desc="Processando Eixos"):
    probs, ids_eixos, status_sugerido = classificar_hibrido(row, labels_ia, df_eixos, label_para_id)

    idx_vencedor_relativo = np.argmax(probs)
    eixo_id_vencedor = ids_eixos[idx_vencedor_relativo]

    # Cálculo estável do Gap de Confiança e da Entropia da Decisão
    scores_ordenados = sorted(probs, reverse=True)
    gap = scores_ordenados[0] - scores_ordenados[1] if len(scores_ordenados) > 1 else 1.0
    entropia = -sum([p * np.log(p + 1e-9) for p in probs])

    # Consolidação do Status de Decisão para Defesa do TCC
    status = status_sugerido
    if status == "IA + DNA Sintático":
        if gap > 0.45:
            status = "Alta Confiança"
        elif row['genero_nome'] == 'Poético/Sapiencial' and idx_vencedor_relativo == 3:
            status = "Narrativa de Moldura (XAI)"
        elif entropia > 1.1:
            status = "Ambiguidade Poética"

    # Preenche o payload respeitando o alinhamento da tabela verso_topico
    results.append({
        'verso_id': int(row['verso_id']),
        'topico_id': int(eixo_id_vencedor),
        'p_exaustao': probs[0],
        'p_transitoriedade': probs[1],
        'p_vazio': probs[2],
        'p_narrativo': probs[3],
        'similaridade_final': probs[idx_vencedor_relativo],
        'margem_dominancia': gap,
        'status_decisao': status,
        'entropia': entropia,
        'gap_confianca': gap
    })

# 4. Persistência Atómica dos Resultados
df_final = pd.DataFrame(results)
cursor = conn.cursor()

print("🧹 Limpando classificações anteriores...")
cursor.execute("DELETE FROM verso_topico")

print("💾 Gravando novos índices existenciais ponderados...")
df_final.to_sql('verso_topico', conn, if_exists='append', index=False)

conn.commit()
conn.close()
print("✨ Célula 5 concluída com sucesso e protegida contra falsos-positivos dialéticos!")

In [ ]:
# Célula 6: Análise de Sentimento Contextualizada com Ponderação Contínua
from pysentimiento import create_analyzer
import pandas as pd
import sqlite3
import numpy as np
from tqdm.auto import tqdm

# 1. Inicializar o Analisador (BERTimbau-based para PT-BR)
print("🚀 Carregando modelo Transformer para Sentimento...")
analyzer = create_analyzer(task="sentiment", lang="pt")

# 2. Busca de dados cruzados
conn = sqlite3.connect(DB_PATH)
query_cruzada = """
    SELECT
        v.id as verso_id,
        v.texto,
        vt.topico_id,
        vl.n_primeira_pessoa,
        vl.is_identidade
    FROM verso v
    JOIN verso_topico vt ON v.id = vt.verso_id
    JOIN verso_limpo vl ON v.id = vl.verso_id
    WHERE v.processar = 'S'
"""
df_input = pd.read_sql_query(query_cruzada, conn)

# Transforma metadados em dicionário indexado por id (O(1) de busca dentro do loop)
meta_lookup = df_input.set_index('verso_id').to_dict('index')

# 3. Execução da análise em lotes
print(f"📊 Analisando carga emocional de {len(df_input)} versículos...")
textos = df_input['texto'].tolist()
verso_ids = df_input['verso_id'].tolist()
sentimentos = []
batch_size = 64
mapa_num = {'POS': 1, 'NEU': 0, 'NEG': -1}

for i in tqdm(range(0, len(textos), batch_size)):
    lote = textos[i:i + batch_size]
    ids_lote = verso_ids[i:i + batch_size]
    preds_lote = analyzer.predict(lote)

    for idx, p in enumerate(preds_lote):
        v_id = ids_lote[idx]
        meta = meta_lookup.get(v_id)

        sc_pos = float(p.probas.get('POS', 0.0))
        sc_neg = float(p.probas.get('NEG', 0.0))
        sc_neu = float(p.probas.get('NEU', 0.0))

        # Ajuste Fino Continuo: Relatos em 1ª pessoa amplificam a variância da polaridade real
        multiplicador_voz = 1.5 if meta and meta['n_primeira_pessoa'] > 0 else 1.0

        # Sentimento base discreto
        sent_base = mapa_num.get(p.output, 0)

        # Cálculo contínuo baseado nas probabilidades (evita o problema de zerar o neutro bruto)
        sentimento_ajustado = float((sc_pos - sc_neg) * multiplicador_voz)

        sentimentos.append({
            'verso_id': int(v_id),
            'label': p.output,
            'sentimento_num': sent_base,
            'sentimento_ajustado': sentimento_ajustado,
            'score_pos': sc_pos,
            'score_neg': sc_neg,
            'score_neu': sc_neu
        })

df_sent = pd.DataFrame(sentimentos)

# 4. Persistência dos Resultados
try:
    cursor = conn.cursor()
    print("🧹 Limpando dados anteriores da tabela 'verso_sentimento'...")
    cursor.execute("DELETE FROM verso_sentimento")

    df_sent.to_sql('verso_sentimento', conn, if_exists='append', index=False)
    conn.commit()
    print("\n✅ Célula 6 concluída! Sentimentos processados de forma contínua.")

    # 5. DIAGNÓSTICO FINAL: CRUZAMENTO EXISTENCIAL DO ANTÍDOTO
    res_final = pd.read_sql_query("""
        SELECT
            t.antidoto_referencia as Eixo_Filosofico,
            COUNT(*) as Total_Versos,
            SUM(CASE WHEN vs.sentimento_num = 1 AND vl.is_identidade = 1 THEN 1 ELSE 0 END) as Definicoes_Fortalecedoras,
            SUM(CASE WHEN vs.sentimento_num = -1 AND vl.n_primeira_pessoa > 0 THEN 1 ELSE 0 END) as Lamentos_Pessoais,
            ROUND(AVG(vs.sentimento_ajustado), 3) as Polaridade_Ponderada_Continua
        FROM verso_topico vt
        JOIN topico t ON vt.topico_id = t.id
        JOIN verso_sentimento vs ON vt.verso_id = vs.verso_id
        JOIN verso_limpo vl ON vt.verso_id = vl.verso_id
        WHERE t.id != 3
        GROUP BY t.antidoto_referencia
        ORDER BY Polaridade_Ponderada_Continua DESC
    """, conn)

    display(res_final)

except Exception as e:
    print(f"❌ Erro na persistência: {e}")
    conn.rollback()
finally:
    conn.close()